In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

# Thiết lập hiển thị đồ thị chuyên nghiệp
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 5]

# Kiểm soát tính tái lập (Reproducibility)
np.random.seed(42)
tf.random.set_seed(42)

# Nạp trực tiếp 2 tập dữ liệu đã được cô lập (KHÔNG DÙNG OS)
print("🎯 Đang kết nối dữ liệu Train và Test...")
train_df_raw = pd.read_csv('../features/train_ml_features.csv')
test_df_raw = pd.read_csv('../features/test_ml_features.csv')

# Ép kiểu Datetime
train_df_raw['order_date'] = pd.to_datetime(train_df_raw['order_date'])
test_df_raw['order_date'] = pd.to_datetime(test_df_raw['order_date'])

# Nối tạm thời để tính toán các biến trễ (Lag) liên tục cho LSTM mà không bị khuyết 7 ngày đầu của Test
df_combined = pd.concat([train_df_raw, test_df_raw]).sort_values(['stock_code', 'order_date']).reset_index(drop=True)

# Thiết lập cửa sổ trượt 7 ngày cho chuỗi thời gian
for i in range(1, 8):
    df_combined[f'lstm_lag_{i}'] = df_combined.groupby('stock_code')['daily_quantity'].shift(i)

# Bỏ các giá trị NaN sinh ra ở những ngày đầu tiên của tập Train
df_combined = df_combined.dropna().reset_index(drop=True)

# =========================================================================
# CHIA TÁCH LẠI NGHIÊM NGẶT (CHỐNG DATA LEAKAGE)
# =========================================================================
# 1. Tách Test: Mọi dữ liệu từ ngày 01/11/2010 trở đi
test_mask = df_combined['order_date'] >= '2010-11-01'
test_df = df_combined[test_mask].sort_values('order_date').reset_index(drop=True)

# 2. Tách Train & Val: Mọi dữ liệu từ 31/10/2010 trở về trước
full_train_df = df_combined[~test_mask].sort_values('order_date').reset_index(drop=True)

# Lấy 80% của full_train làm Train, 20% cuối cùng làm Validation (để LSTM dừng sớm)
split_idx = int(len(full_train_df) * 0.8)
train_df = full_train_df.iloc[:split_idx]
val_df = full_train_df.iloc[split_idx:]

print(f"📊 Kết quả phân rã chuỗi thời gian (Strict Chronological Split):")
print(f"   -> Tập huấn luyện (Train): {train_df.shape[0]:,} mẫu ({train_df['order_date'].min().strftime('%Y-%m-%d')} đến {train_df['order_date'].max().strftime('%Y-%m-%d')})")
print(f"   -> Tập kiểm định (Val):   {val_df.shape[0]:,} mẫu ({val_df['order_date'].min().strftime('%Y-%m-%d')} đến {val_df['order_date'].max().strftime('%Y-%m-%d')})")
print(f"   -> Tập kiểm thử (Test):   {test_df.shape[0]:,} mẫu ({test_df['order_date'].min().strftime('%Y-%m-%d')} đến {test_df['order_date'].max().strftime('%Y-%m-%d')})")

# Khởi tạo từ điển lưu kết quả dự báo ở thang đo sản phẩm gốc
final_predictions = {}

🎯 Đang kết nối dữ liệu Train và Test...
📊 Kết quả phân rã chuỗi thời gian (Strict Chronological Split):
   -> Tập huấn luyện (Train): 140,819 mẫu (2009-12-16 đến 2010-09-17)
   -> Tập kiểm định (Val):   35,205 mẫu (2010-09-17 đến 2010-10-31)
   -> Tập kiểm thử (Test):   35,316 mẫu (2010-11-01 đến 2010-12-09)


In [5]:
# Xác định các cột không tham gia vào quá trình huấn luyện trực tiếp
ignored_cols = ['stock_code', 'order_date', 'daily_quantity']
lstm_lag_cols = [f'lstm_lag_{i}' for i in range(1, 8)]

# 1. Cấu hình đặc trưng cho nhóm mô hình Cây (Giữ nguyên định dạng số gốc cho biến phân loại)
tree_features = [col for col in train_df.columns if col not in ignored_cols and col not in lstm_lag_cols]
X_train_tree = train_df[tree_features]
X_val_tree = val_df[tree_features]
X_test_tree = test_df[tree_features]

y_train = train_df['daily_quantity']
y_val = val_df['daily_quantity']
y_test = test_df['daily_quantity']

# Khôi phục biến mục tiêu y_test về thang đo sản phẩm gốc phục vụ việc tính toán chỉ số sau này
y_test_original = np.expm1(y_test).values

# 2. Cấu hình đặc trưng cho nhóm Tuyến tính (Áp dụng One-Hot Encoding trên tập df_combined)
# Phải biến đổi trên tập tổng để đảm bảo số lượng cột đồng nhất, sau đó mới chia tách lại bằng Mask
df_encoded = pd.get_dummies(df_combined, columns=['month', 'day_of_week'], drop_first=True, dtype=int)
linear_features = [col for col in df_encoded.columns if col not in ignored_cols and col not in lstm_lag_cols]

test_mask_encoded = df_encoded['order_date'] >= '2010-11-01'
X_test_linear = df_encoded[test_mask_encoded].sort_values('order_date').reset_index(drop=True)[linear_features]

full_train_encoded = df_encoded[~test_mask_encoded].sort_values('order_date').reset_index(drop=True)
X_train_linear = full_train_encoded.iloc[:split_idx][linear_features]
X_val_linear = full_train_encoded.iloc[split_idx:][linear_features]

# 3. Khởi tạo danh sách mô hình học máy truyền thống
ml_models = {
    'Multiple Linear': (LinearRegression(), X_train_linear, X_test_linear),
    'Ridge Regression': (Ridge(alpha=1.0), X_train_linear, X_test_linear),
    'Random Forest': (RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1), X_train_tree, X_test_tree),
    'LightGBM': (lgb.LGBMRegressor(n_estimators=150, learning_rate=0.06, random_state=42, verbose=-1, n_jobs=-1), X_train_tree, X_test_tree),
    'XGBoost': (xgb.XGBRegressor(n_estimators=150, max_depth=6, learning_rate=0.06, random_state=42, n_jobs=-1), X_train_tree, X_test_tree)
}

print("[Huấn luyện nhóm mô hình học máy truyền thống]")
for name, (model, X_tr, X_te) in ml_models.items():
    print(f"   -> Đang huấn luyện mô hình: {name}...")
    model.fit(X_tr, y_train)
    pred_log = model.predict(X_te)
    
    # Khôi phục từ thang Log scale về số lượng sản phẩm thực tế và giới hạn giá trị dưới về 0
    final_predictions[name] = np.clip(np.expm1(pred_log), 0, None)

print("✅ Hoàn thành lưu trữ kết quả dự báo của 5 mô hình truyền thống.")

[Huấn luyện nhóm mô hình học máy truyền thống]
   -> Đang huấn luyện mô hình: Multiple Linear...
   -> Đang huấn luyện mô hình: Ridge Regression...
   -> Đang huấn luyện mô hình: Random Forest...
   -> Đang huấn luyện mô hình: LightGBM...
   -> Đang huấn luyện mô hình: XGBoost...
✅ Hoàn thành lưu trữ kết quả dự báo của 5 mô hình truyền thống.


In [7]:
print("[Huấn luyện cấu trúc mạng học sâu tuần tự LSTM]")

# 1. Trích xuất chuỗi thời gian liên tục 7 ngày (Sắp xếp tuần tự từ quá khứ xa đến gần)
lstm_features = [f'lstm_lag_{i}' for i in [7, 6, 5, 4, 3, 2, 1]]

# 2. Chuẩn hóa thang đo dữ liệu (Scaling) đồng bộ cho cả X và Y (Chỉ fit thông số trên tập Train)
scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(train_df[lstm_features])
X_val_scaled = scaler_X.transform(val_df[lstm_features])
X_test_scaled = scaler_X.transform(test_df[lstm_features])

scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(np.array(y_train).reshape(-1, 1)).flatten()
y_val_scaled = scaler_y.transform(np.array(y_val).reshape(-1, 1)).flatten()

# 3. Biến đổi ma trận sang định dạng cấu trúc đầu vào 3D của mạng tuần tự: [Samples, Timesteps=7, Features=1]
X_train_lstm = X_train_scaled.reshape((X_train_scaled.shape[0], 7, 1))
X_val_lstm = X_val_scaled.reshape((X_val_scaled.shape[0], 7, 1))
X_test_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], 7, 1))

# 4. Xây dựng kiến trúc mạng LSTM học chuỗi thời gian
lstm_model = Sequential([
    LSTM(32, activation='tanh', input_shape=(7, 1), return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')

# 5. Cấu hình cơ chế dừng sớm nâng cao chống quá khớp do biến động nhiễu của nhiều SKU
early_stop = EarlyStopping(monitor='val_loss', patience=10, min_delta=1e-4, restore_best_weights=True)

print("   -> Đang thực hiện các chu kỳ tối ưu trọng số (Tối đa 50 Epochs)...")
lstm_model.fit(
    X_train_lstm, y_train_scaled, 
    epochs=50, 
    batch_size=64, # Lựa chọn phổ biến dựa trên thực nghiệm hiệu năng hệ thống
    validation_data=(X_val_lstm, y_val_scaled), 
    callbacks=[early_stop], 
    verbose=1
)

# 6. Dự báo và thực hiện quy trình nghịch đảo thang đo (Inverse Scaling và Expm1)
lstm_pred_scaled = lstm_model.predict(X_test_lstm).flatten()
lstm_pred_log = scaler_y.inverse_transform(lstm_pred_scaled.reshape(-1, 1)).flatten()
final_predictions['LSTM'] = np.clip(np.expm1(lstm_pred_log), 0, None)

# 7. Khởi tạo mô hình cơ sở Naive Baseline từ giá trị lag_1 gốc (Đảm bảo đã khôi phục đơn vị sản phẩm)
# Vì lag_1 nằm trong tập dữ liệu log scale, thực hiện expm1 để đưa về thang đo gốc đối chứng công bằng
final_predictions['Naive Baseline'] = np.clip(np.expm1(X_test_tree['lag_1'].values), 0, None)


# ------------------------------------------------------------------
# ĐÁNH GIÁ TỔNG HỢP VÀ IN BẢNG BÁO CÁO THỐNG KÊ (7 MÔ HÌNH)
# ------------------------------------------------------------------
def safe_mape(y_true, y_pred):
    mask = y_true > 0
    if np.sum(mask) == 0: return 0.0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

print("\n📊 BẢNG ĐÁNH GIÁ HIỆU NĂNG TỔNG THỂ CÁC MÔ HÌNH TRÊN THANG ĐO SẢN PHẨM GỐC:")
print("-" * 115)
print(f"{'Thuật toán / Mô hình':<23} | {'RMSE (Độ lệch bình phương)':<26} | {'MAE (Sai số tuyệt đối)':<22} | {'R² Score':<12} | {'MAPE (% Sai số)'}")
print("-" * 115)

# Quy tắc lựa chọn mô hình (Model Selection Rule): 
# Ưu tiên chọn mô hình có chỉ số sai số RMSE thấp nhất. 
# Nếu chỉ số RMSE tương đương, tiến hành xét chỉ số MAE thấp hơn và R² cao hơn để đánh giá tính ổn định.

for name, pred in final_predictions.items():
    rmse = np.sqrt(mean_squared_error(y_test_original, pred))
    mae = mean_absolute_error(y_test_original, pred)
    r2 = r2_score(y_test_original, pred)
    mape = safe_mape(y_test_original, pred)
    print(f"{name:<23} | {rmse:<26.2f} | {mae:<22.2f} | {r2:<12.4f} | {mape:.2f}%")
print("-" * 115)

# ------------------------------------------------------------------
# XUẤT ĐỒNG BỘ DỮ LIỆU ĐẦU RA CHO STAGE 4
# ------------------------------------------------------------------
df_stage_4 = test_df[['stock_code', 'order_date']].copy()
df_stage_4['actual_quantity'] = np.round(y_test_original).astype(int)

for name, pred in final_predictions.items():
    col_name = f"pred_{name.lower().replace(' ', '_')}"
    df_stage_4[col_name] = np.clip(np.round(pred, 2), 0, None)  # Clip lower to 0

# Thiết lập đường dẫn xuất file trực tiếp cho Stage 4 (Đảm bảo thư mục đã tồn tại trước khi chạy)
output_path = '../features/stage_4_input_predictions.csv'
df_stage_4.to_csv(output_path, index=False)

print(f"\n🎉 KẾT XUẤT FILE THÀNH CÔNG TẠI: '{output_path}'")

[Huấn luyện cấu trúc mạng học sâu tuần tự LSTM]
   -> Đang thực hiện các chu kỳ tối ưu trọng số (Tối đa 50 Epochs)...
Epoch 1/50
2201/2201 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 0.0125 - val_loss: 0.0123
Epoch 2/50
2201/2201 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0123 - val_loss: 0.0123
Epoch 3/50
2201/2201 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0123 - val_loss: 0.0123
Epoch 4/50
2201/2201 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0123 - val_loss: 0.0123
Epoch 5/50
2201/2201 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0123 - val_loss: 0.0123
Epoch 6/50
2201/2201 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - loss: 0.0123 - val_loss: 0.0123
Epoch 7/50
2201/2201 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 0.0123 - val_loss: 0.0123
Epoch 8/50
2201/2201 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0123 - val_loss: 0.0123
Epoch 9/50
2201/2201 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 0.0123 - val_loss: 0.0123
Epoch 10/50
2201/2201 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 0.0123 - val_loss: 0.0123
